In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
#for dirname, _, filenames in os.walk('/kaggle/input'):
#    for filename in filenames:
#        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
# [Cell 2] Setup and config
from collections import Counter
from itertools import combinations
from pathlib import Path
import gc
import hashlib
import json
import random
import shutil
import time
import warnings
import zipfile

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from scipy.stats import rankdata
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from torch.utils.data import Dataset, DataLoader
from torchvision import models

SEED = 42
BACKBONES = ["densenet121", "resnet50", "efficientnet_b0"]
EVAL_BATCH_SIZE = 64
NUM_WORKERS = 2
MIN_POS = 1                   # mirror original tuning code: a label needs both classes
WEIGHT_FRAME_SIZE = None      # use all val_tune images for blending; never use held-out evaluation labels
N_BOOT = 200
CV_FOLDS = 5

DATASETS = {
    "chexpert": {
        # Output order of the models. Must match LABELS in the CheXpert tuning notebook.
        "labels": ["Pleural Effusion", "Cardiomegaly", "Pneumothorax",
                   "Atelectasis", "Consolidation", "Lung Lesion"],
        "label_policy": "uncertain_masked",       # blank -> 0, uncertain (-1) -> masked out
        "dataset_dir": "chexpert-tuning-checkpoint",
        "inner_dir": "multi_backbone_tuning",
        "checkpoint_root": None,  # automatically locate existing extracted Kaggle dataset
        "preprocessed_root": Path("/kaggle/input/notebooks/t1f3001843/dsai-preprocessing"),
        "manifest_rel": "outputs/processed/chexpert_manifest.csv",
        "manifest_glob": "*chexpert*manifest*.csv",
        "resolver": "by_name",
        "eval_sets": ["val_final"],
        "bootstrap_set": "val_final",
    },
    "nih": {
        # Must match LABELS in the NIH tuning notebook.
        "labels": ["Effusion", "Cardiomegaly", "Pneumothorax", "Atelectasis",
                   "Consolidation", "Mass", "Nodule"],
        "label_policy": "binary",
        "dataset_dir": "nih-tuning-checkpoint",
        "inner_dir": "nih_multi_backbone_tuning",
        "checkpoint_root": None,
        "preprocessed_root": Path("/kaggle/input/notebooks/t1f3001843/dsai-nih-preprocessing"),
        "manifest_rel": "outputs/processed/nih_full_manifest.csv",
        "manifest_glob": "nih_full_manifest.csv",
        "resolver": "last3",
        "eval_sets": ["test"],
        "bootstrap_set": "test",
    },
}

OUTPUT_DIR = Path("/kaggle/working/ensemble_results") if Path("/kaggle/working").exists() else Path.cwd() / "ensemble_results"
IMPORT_DIR = OUTPUT_DIR.parent / "imported_checkpoints"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMPORT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()
print(f"Device: {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else ""))


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed()

Device: cuda (Tesla T4)


In [3]:
# [Cell 3] Locate extracted checkpoint directories in the attached Kaggle datasets.
# Expected screenshot:
# /kaggle/input/.../chexpert-tuning-checkpoint/multi_backbone_tuning/{backbone}/best_model.pt
# /kaggle/input/.../nih-tuning-checkpoint/nih_multi_backbone_tuning/{backbone}/best_model.pt
SEARCH_ROOTS = [p for p in [Path("/kaggle/input"), Path("/kaggle/working"), Path.cwd()] if p.exists()]


def has_checkpoints(directory):
    return directory.is_dir() and all((directory / b / "best_model.pt").is_file() for b in BACKBONES)


def find_checkpoint_dir(key, cfg):
    explicit = cfg.get("checkpoint_root")
    if explicit:
        directory = Path(explicit)
        if has_checkpoints(directory):
            return directory
        raise FileNotFoundError(f"{key}: checkpoint_root={directory} does not contain all three best_model.pt files")

    # Match the uploaded dataset name and then its architecture directory; works with
    # both /kaggle/input/<slug>/... and /kaggle/input/datasets/<owner>/<slug>/... mount layouts.
    candidates = []
    for root in SEARCH_ROOTS:
        candidates.extend(root.glob(f"**/{cfg['dataset_dir']}/{cfg['inner_dir']}"))
        candidates.extend(root.glob(f"**/{cfg['inner_dir']}"))
    matches = sorted({p.resolve() for p in candidates if has_checkpoints(p)}, key=str)
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        preferred = [p for p in matches if cfg['dataset_dir'] in p.parts]
        if len(preferred) == 1:
            return preferred[0]
        raise RuntimeError(f"{key}: multiple checkpoint folders found: {matches}. Set checkpoint_root explicitly.")
    raise FileNotFoundError(
        f"{key}: no extracted {cfg['inner_dir']} containing three best_model.pt files under {SEARCH_ROOTS}. "
        "Check that both checkpoint datasets are attached in Kaggle."
    )


CKPT = {}
for key, cfg in DATASETS.items():
    directory = find_checkpoint_dir(key, cfg)
    CKPT[key] = directory
    print(f"{key}: {directory}")
    print({b: f"{(directory / b / 'best_model.pt').stat().st_size / 1e6:.1f} MB" for b in BACKBONES})
    print("split file:", next((str(directory / fn) for fn in ["split_assignment.csv", "split_assignment.csv.gz"]
                                if (directory / fn).exists()), "not included (CheXpert will rebuild exact split)") )


chexpert: /kaggle/input/datasets/t1f3001843/ft-checkpoint/chexpert_tuning_checkpoint (1)/multi_backbone_tuning
{'densenet121': '28.4 MB', 'resnet50': '94.4 MB', 'efficientnet_b0': '16.4 MB'}
split file: /kaggle/input/datasets/t1f3001843/ft-checkpoint/chexpert_tuning_checkpoint (1)/multi_backbone_tuning/split_assignment.csv
nih: /kaggle/input/datasets/t1f3001843/ft-checkpoint/nih_tuning_checkpoint/nih_multi_backbone_tuning
{'densenet121': '28.4 MB', 'resnet50': '94.4 MB', 'efficientnet_b0': '16.4 MB'}
split file: /kaggle/input/datasets/t1f3001843/ft-checkpoint/nih_tuning_checkpoint/nih_multi_backbone_tuning/split_assignment.csv


In [4]:
# [Cell 4] Models and dataset (same as the tuning notebooks)
def adapt_first_conv_to_grayscale(old_conv):
    new_conv = nn.Conv2d(
        1, old_conv.out_channels,
        kernel_size=old_conv.kernel_size, stride=old_conv.stride,
        padding=old_conv.padding, dilation=old_conv.dilation,
        groups=old_conv.groups, bias=old_conv.bias is not None,
    )
    with torch.no_grad():
        new_conv.weight.copy_(old_conv.weight.mean(dim=1, keepdim=True))
        if old_conv.bias is not None:
            new_conv.bias.copy_(old_conv.bias)
    return new_conv


def build_model(backbone, dropout, n_out):
    if backbone == "densenet121":
        model = models.densenet121(weights=None)
        model.features.conv0 = adapt_first_conv_to_grayscale(model.features.conv0)
        model.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(model.classifier.in_features, n_out))
    elif backbone == "resnet50":
        model = models.resnet50(weights=None)
        model.conv1 = adapt_first_conv_to_grayscale(model.conv1)
        model.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(model.fc.in_features, n_out))
    elif backbone == "efficientnet_b0":
        model = models.efficientnet_b0(weights=None)
        model.features[0][0] = adapt_first_conv_to_grayscale(model.features[0][0])
        model.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(model.classifier[-1].in_features, n_out))
    else:
        raise ValueError(f"Unknown backbone: {backbone}")
    return model


class ShardDataset(Dataset):
    """Eval-only shard dataset. Returns (image, targets, mask); mask is all ones when the frame has no mask columns."""
    # CheXpert tuning uses raw 0..1 grayscale; keep NIH standardization as previously implemented.
    MEAN, STD = 0.449, 0.226

    def __init__(self, frame, labels, dataset_key):
        self.dataset_key = dataset_key
        self.frame = frame.reset_index(drop=True)
        self.paths = self.frame["shard_path"].astype(str).to_numpy()
        self.positions = self.frame["shard_index"].astype(int).to_numpy()
        self.targets = self.frame[labels].to_numpy(dtype=np.float32)
        mask_cols = [f"{c}__mask" for c in labels]
        self.masks = (self.frame[mask_cols].to_numpy(dtype=np.float32)
                      if all(c in self.frame.columns for c in mask_cols) else np.ones_like(self.targets))
        self.shard_cache = {}

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        path, pos = self.paths[index], int(self.positions[index])
        if path not in self.shard_cache:
            self.shard_cache[path] = np.load(path, mmap_mode="r", allow_pickle=False)
        shard = self.shard_cache[path]
        if not 0 <= pos < len(shard):
            raise IndexError(f"shard_index {pos} is outside shard of length {len(shard)}: {path}")
        image = np.asarray(shard[pos])
        if image.ndim != 2:
            raise ValueError(f"Expected grayscale (H, W) image; got {image.shape}")
        image = image.astype(np.float32)
        if image.max() > 1.0:
            image /= 255.0
        image = torch.from_numpy(image.copy()).unsqueeze(0)
        if self.dataset_key == "nih":
            image = (image - self.MEAN) / self.STD
        return image, torch.from_numpy(self.targets[index]), torch.from_numpy(self.masks[index])


def make_loader(frame, labels, dataset_key):
    return DataLoader(ShardDataset(frame, labels, dataset_key), batch_size=EVAL_BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
                      persistent_workers=(NUM_WORKERS > 0))


def load_best_model(key, backbone, n_out):
    ckpt = torch.load(CKPT[key] / backbone / "best_model.pt", map_location="cpu", weights_only=True)
    model = build_model(backbone, ckpt["config"]["dropout"], n_out)
    model.load_state_dict(ckpt["model_state_dict"])   # fails loudly if the number of outputs differs
    return model.to(DEVICE).eval()


@torch.no_grad()
def collect_outputs(model, frame, labels, dataset_key):
    logits, targets, masks = [], [], []
    for images, y, m in make_loader(frame, labels, dataset_key):
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            out = model(images.to(DEVICE, non_blocking=True))
        logits.append(out.float().cpu())
        targets.append(y)
        masks.append(m)
    return torch.cat(logits).numpy(), torch.cat(targets).numpy(), torch.cat(masks).numpy()


print("Models and dataset ready.")

Models and dataset ready.


In [5]:
# [Cell 5] Ensemble methods, metrics and bootstrap
EPS = 1e-3

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def auroc_per_label(scores, targets, masks):
    out = np.full(targets.shape[1], np.nan)
    for j in range(targets.shape[1]):
        keep = masks[:, j].astype(bool)
        y = targets[keep, j]
        if (y == 1).sum() < MIN_POS or (y == 0).sum() < MIN_POS:
            continue
        out[j] = roc_auc_score(y, scores[keep, j])
    return out

def macro(values):
    finite = values[np.isfinite(values)]
    return float(finite.mean()) if finite.size else float("nan")

def rank_normalise(Z):
    out = np.empty(Z.shape, dtype=np.float64)
    for b in range(Z.shape[0]):
        out[b] = rankdata(Z[b], axis=0) / Z.shape[1]
    return out

def fixed(score_fn):
    def fit(Z, y, m):
        return score_fn
    return fit


def fit_weighted_global(Z, y, m):
    aucs = np.array([macro(auroc_per_label(Z[b], y, m)) for b in range(Z.shape[0])])
    w = np.clip(np.nan_to_num(aucs, nan=0.5) - 0.5, EPS, None)
    w = w / w.sum()

    def predict(Z2):
        return np.tensordot(w, Z2, axes=(0, 0))
    predict.info = {"weights": w.tolist()}
    return predict


def fit_weighted_per_label(Z, y, m):
    aucs = np.stack([auroc_per_label(Z[b], y, m) for b in range(Z.shape[0])])
    W = np.clip(np.nan_to_num(aucs, nan=0.5) - 0.5, EPS, None)
    W = W / W.sum(axis=0, keepdims=True)

    def predict(Z2):
        return (W[:, None, :] * Z2).sum(axis=0)
    predict.info = {"weights": W.tolist()}
    return predict


def _fit_logreg(X, y, C):
    if (y == 1).sum() < 2 or (y == 0).sum() < 2:
        return None
    return LogisticRegression(C=C, max_iter=1000).fit(X, y)


def fit_stack_own(Z, y, m, C=1.0):
    B, _, L = Z.shape
    fitted = [_fit_logreg(Z[:, m[:, j].astype(bool), j].T, y[m[:, j].astype(bool), j], C) for j in range(L)]

    def predict(Z2):
        out = np.empty(Z2.shape[1:], dtype=np.float64)
        for j, mdl in enumerate(fitted):
            X = Z2[:, :, j].T
            out[:, j] = X.mean(axis=1) if mdl is None else mdl.decision_function(X)
        return out
    predict.info = {"coef": [None if mdl is None else mdl.coef_[0].tolist() for mdl in fitted]}
    return predict


def fit_stack_all(Z, y, m, C=0.1):
    B, n, L = Z.shape
    X_fit = Z.transpose(1, 0, 2).reshape(n, B * L)
    fitted = [_fit_logreg(X_fit[m[:, j].astype(bool)], y[m[:, j].astype(bool), j], C) for j in range(L)]

    def predict(Z2):
        X = Z2.transpose(1, 0, 2).reshape(Z2.shape[1], B * L)
        out = np.empty(Z2.shape[1:], dtype=np.float64)
        for j, mdl in enumerate(fitted):
            out[:, j] = Z2[:, :, j].mean(axis=0) if mdl is None else mdl.decision_function(X)
        return out
    return predict


METHODS = {
    "mean_prob": fixed(lambda Z: sigmoid(Z).mean(axis=0)),
    "geo_mean_prob": fixed(lambda Z: np.exp(np.log(np.clip(sigmoid(Z), 1e-12, 1.0)).mean(axis=0))),
    "median_prob": fixed(lambda Z: np.median(sigmoid(Z), axis=0)),
    "mean_logit": fixed(lambda Z: Z.mean(axis=0)),
    "rank_mean": fixed(lambda Z: rank_normalise(Z).mean(axis=0)),
    "weighted_logit": fit_weighted_global,
    "weighted_logit_per_label": fit_weighted_per_label,
    "stack_own": fit_stack_own,
    "stack_all": fit_stack_all,
}


def oof_scores(fit, Z, y, m, groups, n_splits=CV_FOLDS):
    """Patient-grouped cross-fitted scores, so fitted methods are not judged on their own fitting data."""
    out = np.zeros(Z.shape[1:], dtype=np.float64)
    for train_idx, hold_idx in GroupKFold(n_splits=n_splits).split(np.arange(Z.shape[1]), groups=groups):
        predict = fit(Z[:, train_idx], y[train_idx], m[train_idx])
        out[hold_idx] = predict(Z[:, hold_idx])
    return out


def paired_bootstrap(score_dict, y, m, patient_ids, n_boot=N_BOOT, seed=SEED):
    """Resample PATIENTS; return model names and an (n_boot, n_models) array of macro AUROCs (same draws for all models)."""
    codes, uniques = pd.factorize(pd.Series(np.asarray(patient_ids).astype(str)))
    order = np.argsort(codes, kind="stable")
    bounds = np.searchsorted(codes[order], np.arange(len(uniques) + 1))
    patient_rows = [order[bounds[i]:bounds[i + 1]] for i in range(len(uniques))]
    names = list(score_dict)
    rng = np.random.default_rng(seed)
    draws = np.full((n_boot, len(names)), np.nan)
    for d in range(n_boot):
        idx = np.concatenate([patient_rows[i] for i in rng.integers(0, len(patient_rows), len(patient_rows))])
        for k, name in enumerate(names):
            draws[d, k] = macro(auroc_per_label(score_dict[name][idx], y[idx], m[idx]))
    return names, draws


print("Methods:", ", ".join(METHODS))

Methods: mean_prob, geo_mean_prob, median_prob, mean_logit, rank_mean, weighted_logit, weighted_logit_per_label, stack_own, stack_all


In [6]:
# [Cell 6] Rebuild split frames for a dataset from its zip + manifest
def build_shard_resolver(root, mode, manifest_dir):
    all_npy = list(root.rglob("*.npy"))
    if not all_npy:
        raise FileNotFoundError(f"No .npy shards found under {root}. Is the preprocessing output attached as an input?")
    if mode == "last3":                 # NIH: <export>/<split>/images_#####.npy (file names repeat across splits)
        index = {p.parts[-3:]: p for p in all_npy}

        def resolve(value):
            candidate = Path(str(value))
            if candidate.is_file():
                return str(candidate)
            hit = index.get(candidate.parts[-3:])
            return str(hit) if hit is not None else None
    else:                               # CheXpert: same file-name logic as the tuning notebook
        by_name = {}
        for p in all_npy:
            by_name.setdefault(p.name, []).append(p)

        def resolve(value):
            candidate = Path(str(value))
            for path in [candidate, root / candidate, manifest_dir / candidate]:
                if path.is_file():
                    return str(path)
            matches = by_name.get(candidate.name, [])
            chexpert = [p for p in matches if "chexpert" in str(p).lower()]
            if len(chexpert) == 1:
                return str(chexpert[0])
            if len(matches) == 1:
                return str(matches[0])
            return None
    return resolve


def prepare_labels(frame, cfg):
    frame = frame.copy()
    for c in cfg["labels"]:
        raw = pd.to_numeric(frame[c], errors="coerce")
        if cfg["label_policy"] == "uncertain_masked":
            frame[f"{c}__mask"] = (raw.notna() & (raw != -1)).astype(np.float32)
            frame[c] = raw.eq(1).astype(np.float32)
        else:
            if not raw.isin([0, 1]).all():
                raise ValueError(f"Label column {c} has values other than 0/1.")
            frame[c] = raw.astype(np.float32)
    return frame


def sample_frame(df, n):
    return df.sample(n=min(n, len(df)), random_state=SEED).reset_index(drop=True)   # same call as the tuning notebooks


def locate_manifest(cfg):
    root = cfg["preprocessed_root"]
    requested = root / cfg["manifest_rel"]
    if requested.exists():
        return root, requested
    # Notebook input folder names may differ from the Kaggle UI display name.
    for search_root in [Path("/kaggle/input/notebooks"), Path("/kaggle/input")]:
        if search_root.exists():
            hits = sorted(search_root.rglob(cfg["manifest_glob"]))
            if len(hits) == 1:
                hit = hits[0]
                root = next((parent for parent in hit.parents if parent.name.lower().startswith("dsai") ), hit.parent)
                return root, hit
            if len(hits) > 1:
                preferred = [hit for hit in hits if ("nih" if cfg["resolver"] == "last3" else "chexpert") in str(hit).lower()]
                if len(preferred) == 1:
                    hit = preferred[0]
                    root = next((parent for parent in hit.parents if parent.name.lower().startswith("dsai")), hit.parent)
                    return root, hit
    raise FileNotFoundError(f"Cannot find {cfg['manifest_glob']} in Kaggle preprocessing inputs")


EXTRA_VALID_PATIENTS = 6000   # must match the tuning notebook
VALID_SEED = 42               # must match the tuning notebook


def rebuild_chexpert_splits(manifest):
    split = manifest["split"].astype(str).str.lower()
    train = manifest[split.eq("train")]
    valid = manifest[split.isin(["valid", "validation"])]

    # Same patient-level carve-out from train as in the tuning notebook
    rng = np.random.default_rng(VALID_SEED)
    train_patient_ids = np.array(sorted(train["patient_id"].astype(str).unique()))
    extra_patients = set(rng.choice(train_patient_ids, size=EXTRA_VALID_PATIENTS, replace=False))
    extra_mask = train["patient_id"].astype(str).isin(extra_patients)
    valid_all = pd.concat([valid, train[extra_mask]], ignore_index=True)
    train = train[~extra_mask]

    selected = valid_all.sample(n=min(19100, len(valid_all)), random_state=SEED).reset_index(drop=True)
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=SEED)
    tune_idx, final_idx = next(splitter.split(selected, groups=selected["patient_id"]))
    frames = {"tune": prepare_labels(selected.iloc[tune_idx].reset_index(drop=True), DATASETS["chexpert"]),
              "val_final": prepare_labels(selected.iloc[final_idx].reset_index(drop=True), DATASETS["chexpert"])}
    train_patients = set(train["patient_id"].astype(str))
    assert not (train_patients & set(frames["tune"]["patient_id"]))
    assert not (train_patients & set(frames["val_final"]["patient_id"]))
    return frames


def load_frames(key):
    cfg = DATASETS[key]
    root, manifest_path = locate_manifest(cfg)
    print(f"[{key}] manifest: {manifest_path}")
    manifest = pd.read_csv(manifest_path)
    required = {"patient_id", "shard_index", "shard_path", *cfg["labels"]}
    if key == "chexpert":
        required.add("split")
    missing = required - set(manifest.columns)
    if missing:
        raise ValueError(f"Manifest missing columns: {sorted(missing)}")
    resolve = build_shard_resolver(root, cfg["resolver"], manifest_path.parent)
    manifest["shard_path"] = manifest["shard_path"].map(resolve)
    if manifest["shard_path"].isna().any():
        raise FileNotFoundError(f"Could not resolve {manifest['shard_path'].isna().sum()} image rows to shards")
    manifest["shard_index"] = manifest["shard_index"].astype(int)
    manifest["patient_id"] = manifest["patient_id"].astype(str)

    assignment_path = next((CKPT[key] / fn for fn in ["split_assignment.csv", "split_assignment.csv.gz"]
                            if (CKPT[key] / fn).is_file()), None)
    if key == "chexpert":
        frames = rebuild_chexpert_splits(manifest)
        if assignment_path is not None:
            print(f"[{key}] saved assignment exists ({assignment_path.name}); using exact tuning-notebook split instead")
    else:
        if assignment_path is None:
            raise FileNotFoundError(f"[{key}] missing split_assignment.csv inside {CKPT[key]}")
        assignment = pd.read_csv(assignment_path, dtype={"patient_id": str})
        if "set" not in assignment.columns:
            raise ValueError(f"[{key}] assignment requires 'set' column; got {list(assignment.columns)}")
        assignment["shard_path"] = assignment["shard_path"].map(resolve)
        if assignment["shard_path"].isna().any():
            raise FileNotFoundError(f"[{key}] failed resolving split-assignment shard paths")
        assignment["shard_index"] = assignment["shard_index"].astype(int)
        print(f"[{key}] splits: {assignment['set'].value_counts().to_dict()}")
        frames = {}
        for set_name in ["tune", *cfg["eval_sets"]]:
            keys = assignment.loc[assignment["set"] == set_name, ["shard_path", "shard_index"]]
            if keys.empty:
                raise ValueError(f"[{key}] split {set_name!r} is empty. Available splits: {assignment['set'].unique()}")
            merged = keys.merge(manifest, on=["shard_path", "shard_index"], how="left", indicator=True, validate="many_to_one")
            if (merged["_merge"] != "both").any():
                raise ValueError(f"[{key}] {set_name} images missing from manifest")
            frames[set_name] = prepare_labels(merged.drop(columns="_merge"), cfg)

    for a, b in combinations(frames, 2):
        shared = set(frames[a]["patient_id"]) & set(frames[b]["patient_id"])
        if shared:
            raise ValueError(f"[{key}] patient leakage between {a} and {b}: {len(shared)}")
    print(f"[{key}] evaluation sizes: { {name: len(frame) for name, frame in frames.items()} }")
    return frames


print("Frame loader ready.")

Frame loader ready.


In [7]:
# [Cell 7] Logits (cached), reproduction check, analysis
RUNS = {}


def frame_signature(frame):
    text = (frame["shard_path"].astype(str) + "#" + frame["shard_index"].astype(str)).str.cat(sep="|")
    return hashlib.md5(text.encode()).hexdigest()


def prepare_dataset(key):
    cfg = DATASETS[key]
    labels = cfg["labels"]
    frames = load_frames(key)
    weight_frame = frames["tune"] if WEIGHT_FRAME_SIZE is None else sample_frame(frames["tune"], WEIGHT_FRAME_SIZE)
    all_sets = {"weights": weight_frame, **{s: frames[s] for s in cfg["eval_sets"]}}

    out_dir = OUTPUT_DIR / key
    out_dir.mkdir(parents=True, exist_ok=True)
    meta = {"signatures": {n: frame_signature(f) for n, f in all_sets.items()}, "backbones": BACKBONES, "labels": labels, "checkpoint_fingerprints": {b: [int((CKPT[key] / b / "best_model.pt").stat().st_size), int((CKPT[key] / b / "best_model.pt").stat().st_mtime_ns)] for b in BACKBONES}}
    cache_path = out_dir / "logits.npz"

    store = {}
    if cache_path.exists():
        cached = np.load(cache_path, allow_pickle=False)
        if json.loads(str(cached["meta"])) == meta:
            for name in all_sets:
                store[name] = {"logits": np.stack([cached[f"{name}__{b}"] for b in BACKBONES]),
                               "targets": cached[f"{name}__targets"], "masks": cached[f"{name}__masks"]}
            print(f"[{key}] loaded cached logits")
    if not store:
        per_set = {n: {"logits": [], "targets": None, "masks": None} for n in all_sets}
        for backbone in BACKBONES:
            t0 = time.time()
            model = load_best_model(key, backbone, len(labels))
            for name, frame in all_sets.items():
                lg, y, m = collect_outputs(model, frame, labels, key)
                if per_set[name]["targets"] is None:
                    per_set[name]["targets"], per_set[name]["masks"] = y, m
                per_set[name]["logits"].append(lg)
            del model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            print(f"[{key}] scored with {backbone} in {time.time() - t0:.0f}s")
        store = {n: {"logits": np.stack(v["logits"]), "targets": v["targets"], "masks": v["masks"]} for n, v in per_set.items()}
        arrays = {"meta": np.array(json.dumps(meta))}
        for name, v in store.items():
            arrays[f"{name}__targets"], arrays[f"{name}__masks"] = v["targets"], v["masks"]
            for i, b in enumerate(BACKBONES):
                arrays[f"{name}__{b}"] = v["logits"][i]
        np.savez_compressed(cache_path, **arrays)

    RUNS[key] = {"cfg": cfg, "labels": labels, "frames": frames, "weight_frame": weight_frame,
                 "eval_frames": {s: frames[s] for s in cfg["eval_sets"]}, "store": store, "out_dir": out_dir}
    for name, v in store.items():
        print(f"[{key}] {name:9s} n={v['logits'].shape[1]:>7,}  logits {v['logits'].shape}")
    reproduction_check(key)


def reproduction_check(key):
    """Recompute each backbone's AUROC here and compare with what the tuning notebook recorded in the zip."""
    run = RUNS[key]
    ref_path = CKPT[key] / "architecture_comparison.csv"
    if not ref_path.exists():
        print(f"[{key}] no architecture_comparison.csv in the zip; reproduction check skipped.")
        return
    ref = pd.read_csv(ref_path).set_index("backbone")
    pairs = {"val_final": "val_final_macro_auroc", "test": "test_macro_auroc"}
    rows = []
    for set_name, column in pairs.items():
        if set_name not in run["store"] or column not in ref.columns:
            continue
        s = run["store"][set_name]
        for i, backbone in enumerate(BACKBONES):
            mine = macro(auroc_per_label(s["logits"][i], s["targets"], s["masks"]))
            recorded = float(ref.loc[backbone, column])
            rows.append({"set": set_name, "backbone": backbone, "recorded": recorded, "recomputed": mine,
                         "abs_diff": abs(mine - recorded) if np.isfinite(recorded) else np.nan})
    table = pd.DataFrame(rows)
    display(table.round(5))
    if table.empty:
        print(f"[{key}] no matching metric columns for reproduction check; skipped.")
        return
    worst = table["abs_diff"].max()
    if worst < 2e-3:
        print(f"[{key}] reproduction OK (max |diff| = {worst:.5f}): splits, labels and preprocessing match the tuning run.")
    else:
        print(f"[{key}] WARNING: max |diff| = {worst:.4f}. Check the label order in Cell 2, the preprocessing mount, "
              "and checkpoint compatibility.")


def analyse(key):
    run = RUNS[key]
    labels, store, cfg = run["labels"], run["store"], run["cfg"]
    Zw, yw, mw = store["weights"]["logits"], store["weights"]["targets"], store["weights"]["masks"]
    groups = run["weight_frame"]["patient_id"].astype(str).to_numpy()

    single_val = {b: macro(auroc_per_label(Zw[i], yw, mw)) for i, b in enumerate(BACKBONES)}
    ensemble_val = {name: macro(auroc_per_label(oof_scores(fit, Zw, yw, mw, groups), yw, mw))
                    for name, fit in METHODS.items()}
    best_single = max(single_val, key=single_val.get)
    best_ensemble = max(ensemble_val, key=ensemble_val.get)
    predictors = {name: fit(Zw, yw, mw) for name, fit in METHODS.items()}

    scores, tables = {}, {}
    for set_name in cfg["eval_sets"]:
        Z, y, m = store[set_name]["logits"], store[set_name]["targets"], store[set_name]["masks"]
        s = {b: Z[i] for i, b in enumerate(BACKBONES)}
        s.update({name: predictors[name](Z) for name in METHODS})
        scores[set_name] = s
        rows = []
        for name, sc in s.items():
            per_label = auroc_per_label(sc, y, m)
            single = name in single_val
            rows.append({"model": name, "kind": "single" if single else "ensemble",
                         "val_macro_auroc": single_val[name] if single else ensemble_val[name],
                         "macro_auroc": macro(per_label), **dict(zip(labels, per_label))})
        table = pd.DataFrame(rows)
        table["selected"] = table["model"].isin([best_single, best_ensemble])
        tables[set_name] = table

    bset = cfg["bootstrap_set"]
    t0 = time.time()
    names, draws = paired_bootstrap(scores[bset], store[bset]["targets"], store[bset]["masks"],
                                    run["eval_frames"][bset]["patient_id"].to_numpy())
    col = {n: i for i, n in enumerate(names)}
    rows = []
    for name in names:
        lo, hi = np.nanpercentile(draws[:, col[name]], [2.5, 97.5])
        row = {"model": name, "macro_ci_low": lo, "macro_ci_high": hi}
        if name not in single_val:
            diff = draws[:, col[name]] - draws[:, col[best_single]]
            row.update({"diff_vs_best_single": float(np.nanmean(diff)),
                        "diff_ci_low": float(np.nanpercentile(diff, 2.5)),
                        "diff_ci_high": float(np.nanpercentile(diff, 97.5)),
                        "p_better_than_best_single": float(np.nanmean(diff > 0))})
        rows.append(row)
    boot_table = pd.DataFrame(rows)
    boot_table.insert(1, "macro_auroc", boot_table["model"].map(tables[bset].set_index("model")["macro_auroc"]))
    print(f"[{key}] bootstrap ({N_BOOT} draws) took {time.time() - t0:.0f}s")

    Z = store[bset]["logits"]
    div_rows = []
    for (i, a), (j, b) in combinations(enumerate(BACKBONES), 2):
        corr = [np.corrcoef(rankdata(Z[i][:, l]), rankdata(Z[j][:, l]))[0, 1] for l in range(len(labels))]
        div_rows.append({"pair": f"{a} vs {b}", "mean_rank_corr": float(np.mean(corr)), **dict(zip(labels, corr))})
    diversity = pd.DataFrame(div_rows)

    result = {"single_val": single_val, "ensemble_val": ensemble_val, "best_single": best_single,
              "best_ensemble": best_ensemble, "tables": tables, "boot_table": boot_table,
              "diversity": diversity, "predictors": predictors}
    run["result"] = result

    out_dir = run["out_dir"]
    for set_name, table in tables.items():
        table.to_csv(out_dir / f"results_{set_name}.csv", index=False)
    boot_table.to_csv(out_dir / f"bootstrap_{bset}.csv", index=False)
    diversity.to_csv(out_dir / "backbone_rank_correlation.csv", index=False)
    (out_dir / "fit_info.json").write_text(json.dumps({
        "labels": labels, "backbones": BACKBONES, "single_val_macro_auroc": single_val,
        "ensemble_val_macro_auroc_cv": ensemble_val, "best_single_by_val": best_single,
        "best_ensemble_by_val": best_ensemble,
        "fit_info": {n: getattr(p, "info", None) for n, p in predictors.items()}}, indent=2, default=float))

    print(f"\n[{key}] best single on the weight sample : {best_single} ({single_val[best_single]:.4f})")
    print(f"[{key}] best ensemble (cross-fitted)     : {best_ensemble} ({ensemble_val[best_ensemble]:.4f})")
    for set_name, table in tables.items():
        print(f"\n=== {key} / {set_name}: per-label AUROC (selected rows marked) ===")
        display(table.sort_values("macro_auroc", ascending=False).round(4))
    print(f"\n=== {key}: paired bootstrap on '{bset}' (differences are vs best single = {best_single}) ===")
    display(boot_table.round(4))
    print(f"=== {key}: rank correlation between backbones on '{bset}' (high = little to gain from averaging) ===")
    display(diversity.round(3))


print("Pipeline functions ready.")

Pipeline functions ready.


In [8]:
# [Cell 8] CheXpert: rebuild splits, score with the 3 checkpoints, reproduction check
prepare_dataset("chexpert")

[chexpert] manifest: /kaggle/input/notebooks/t1f3001843/dsai-preprocessing/outputs/processed/chexpert_manifest.csv
[chexpert] saved assignment exists (split_assignment.csv); using exact tuning-notebook split instead
[chexpert] evaluation sizes: {'tune': 8929, 'val_final': 9376}
[chexpert] scored with densenet121 in 165s
[chexpert] scored with resnet50 in 25s
[chexpert] scored with efficientnet_b0 in 17s
[chexpert] weights   n=  8,929  logits (3, 8929, 6)
[chexpert] val_final n=  9,376  logits (3, 9376, 6)


""


[chexpert] no matching metric columns for reproduction check; skipped.


In [9]:
# [Cell 9] CheXpert: ensemble comparison, paired bootstrap, backbone diversity
analyse("chexpert")

[chexpert] bootstrap (200 draws) took 28s

[chexpert] best single on the weight sample : densenet121 (0.8111)
[chexpert] best ensemble (cross-fitted)     : stack_all (0.8234)

=== chexpert / val_final: per-label AUROC (selected rows marked) ===


,model,kind,val_macro_auroc,macro_auroc,Pleural Effusion,Cardiomegaly,Pneumothorax,Atelectasis,Consolidation,Lung Lesion,selected
11,stack_all,ensemble,0.8234,0.8305,0.9399,0.9226,0.7911,0.7915,0.9157,0.6220,True
8,weighted_logit,ensemble,0.8200,0.8142,0.9328,0.8879,0.7834,0.7912,0.8957,0.5941,False
9,weighted_logit_per_label,ensemble,0.8216,0.8140,0.9322,0.8882,0.7826,0.7927,0.8956,0.5927,False
10,stack_own,ensemble,0.8173,0.8138,0.9352,0.8887,0.7834,0.7964,0.9016,0.5773,False
6,mean_logit,ensemble,0.8186,0.8130,0.9316,0.8868,0.7820,0.7916,0.8931,0.5929,False
4,geo_mean_prob,ensemble,0.8179,0.8123,0.9292,0.8864,0.7825,0.7905,0.8930,0.5924,False
7,rank_mean,ensemble,0.8163,0.8108,0.9317,0.8790,0.7810,0.7918,0.8977,0.5839,False
3,mean_prob,ensemble,0.8108,0.8081,0.9317,0.8834,0.7492,0.7937,0.8922,0.5985,False
5,median_prob,ensemble,0.8195,0.8064,0.9287,0.8847,0.7711,0.7810,0.8929,0.5798,False
0,densenet121,single,0.8111,0.7962,0.9262,0.8781,0.7407,0.7836,0.8886,0.5601,True



=== chexpert: paired bootstrap on 'val_final' (differences are vs best single = densenet121) ===


,model,macro_auroc,macro_ci_low,macro_ci_high,diff_vs_best_single,diff_ci_low,diff_ci_high,p_better_than_best_single
0,densenet121,0.7962,0.7821,0.8104,NaN,NaN,NaN,NaN
1,resnet50,0.7904,0.7762,0.8077,NaN,NaN,NaN,NaN
2,efficientnet_b0,0.7503,0.7319,0.7667,NaN,NaN,NaN,NaN
3,mean_prob,0.8081,0.7930,0.8238,0.0120,0.0023,0.0223,0.990
4,geo_mean_prob,0.8123,0.7971,0.8275,0.0161,0.0063,0.0261,1.000
5,median_prob,0.8064,0.7903,0.8207,0.0102,0.0017,0.0187,0.990
6,mean_logit,0.8130,0.7978,0.8282,0.0168,0.0071,0.0265,1.000
7,rank_mean,0.8108,0.7964,0.8254,0.0147,0.0049,0.0241,0.995
8,weighted_logit,0.8142,0.7990,0.8290,0.0180,0.0090,0.0267,1.000
9,weighted_logit_per_label,0.8140,0.7989,0.8294,0.0178,0.0105,0.0252,1.000


=== chexpert: rank correlation between backbones on 'val_final' (high = little to gain from averaging) ===


,pair,mean_rank_corr,Pleural Effusion,Cardiomegaly,Pneumothorax,Atelectasis,Consolidation,Lung Lesion
0,densenet121 vs resnet50,0.655,0.843,0.771,0.497,0.681,0.739,0.400
1,densenet121 vs efficientnet_b0,0.538,0.781,0.541,0.339,0.676,0.584,0.306
2,resnet50 vs efficientnet_b0,0.557,0.753,0.515,0.408,0.581,0.635,0.447


In [10]:
# [Cell 10] NIH: rebuild splits, score with the 3 checkpoints, reproduction check
prepare_dataset("nih")

[nih] manifest: /kaggle/input/notebooks/t1f3001843/dsai-nih-preprocessing/outputs/processed/nih_full_manifest.csv
[nih] splits: {'train': 22319, 'test': 11778, 'tune': 2726}
[nih] evaluation sizes: {'tune': 2726, 'test': 11778}
[nih] scored with densenet121 in 34s
[nih] scored with resnet50 in 20s
[nih] scored with efficientnet_b0 in 12s
[nih] weights   n=  2,726  logits (3, 2726, 7)
[nih] test      n= 11,778  logits (3, 11778, 7)


,set,backbone,recorded,recomputed,abs_diff
0,test,densenet121,0.79193,0.79193,0.0
1,test,resnet50,0.78891,0.78891,0.0
2,test,efficientnet_b0,0.79221,0.79221,0.0


[nih] reproduction OK (max |diff| = 0.00000): splits, labels and preprocessing match the tuning run.


In [11]:
# [Cell 11] NIH: ensemble comparison, paired bootstrap, backbone diversity
analyse("nih")

[nih] bootstrap (200 draws) took 56s

[nih] best single on the weight sample : densenet121 (0.8213)
[nih] best ensemble (cross-fitted)     : weighted_logit_per_label (0.8349)

=== nih / test: per-label AUROC (selected rows marked) ===


,model,kind,val_macro_auroc,macro_auroc,Effusion,Cardiomegaly,Pneumothorax,Atelectasis,Consolidation,Mass,Nodule,selected
6,mean_logit,ensemble,0.8349,0.8069,0.8076,0.9153,0.8663,0.7506,0.7198,0.8104,0.7784,False
8,weighted_logit,ensemble,0.8349,0.8069,0.8076,0.9153,0.8663,0.7506,0.7198,0.8104,0.7784,False
9,weighted_logit_per_label,ensemble,0.8349,0.8069,0.8076,0.9154,0.8661,0.7506,0.7198,0.8103,0.7784,True
4,geo_mean_prob,ensemble,0.8347,0.8067,0.8072,0.9152,0.8661,0.7502,0.7199,0.8101,0.7783,False
10,stack_own,ensemble,0.8328,0.8066,0.8072,0.9157,0.8662,0.7492,0.7188,0.8101,0.7792,False
3,mean_prob,ensemble,0.8343,0.8066,0.8076,0.9151,0.8648,0.7507,0.7187,0.8103,0.7789,False
7,rank_mean,ensemble,0.8331,0.8064,0.8074,0.9142,0.8658,0.7504,0.7195,0.8094,0.7780,False
11,stack_all,ensemble,0.8272,0.8046,0.8064,0.9105,0.8638,0.7408,0.7215,0.8065,0.7824,False
5,median_prob,ensemble,0.8313,0.8037,0.8058,0.9115,0.8611,0.7487,0.7167,0.8076,0.7744,False
2,efficientnet_b0,single,0.8156,0.7922,0.7941,0.8934,0.8541,0.7352,0.7074,0.7954,0.7659,False



=== nih: paired bootstrap on 'test' (differences are vs best single = densenet121) ===


,model,macro_auroc,macro_ci_low,macro_ci_high,diff_vs_best_single,diff_ci_low,diff_ci_high,p_better_than_best_single
0,densenet121,0.7919,0.7850,0.7985,NaN,NaN,NaN,NaN
1,resnet50,0.7889,0.7821,0.7964,NaN,NaN,NaN,NaN
2,efficientnet_b0,0.7922,0.7859,0.7989,NaN,NaN,NaN,NaN
3,mean_prob,0.8066,0.8003,0.8127,0.0147,0.0123,0.0174,1.0
4,geo_mean_prob,0.8067,0.8005,0.8130,0.0148,0.0126,0.0175,1.0
5,median_prob,0.8037,0.7976,0.8099,0.0118,0.0094,0.0140,1.0
6,mean_logit,0.8069,0.8007,0.8132,0.0150,0.0127,0.0176,1.0
7,rank_mean,0.8064,0.8003,0.8127,0.0145,0.0122,0.0170,1.0
8,weighted_logit,0.8069,0.8007,0.8132,0.0150,0.0127,0.0176,1.0
9,weighted_logit_per_label,0.8069,0.8007,0.8132,0.0150,0.0127,0.0176,1.0


=== nih: rank correlation between backbones on 'test' (high = little to gain from averaging) ===


,pair,mean_rank_corr,Effusion,Cardiomegaly,Pneumothorax,Atelectasis,Consolidation,Mass,Nodule
0,densenet121 vs resnet50,0.823,0.860,0.789,0.793,0.856,0.821,0.803,0.836
1,densenet121 vs efficientnet_b0,0.793,0.862,0.756,0.731,0.831,0.777,0.784,0.813
2,resnet50 vs efficientnet_b0,0.789,0.859,0.739,0.814,0.822,0.745,0.762,0.783


In [12]:
# [Cell 12] Combined summary + export
summary_rows, method_columns = [], {}
for key, run in RUNS.items():
    res = run["result"]
    bset = run["cfg"]["bootstrap_set"]
    macro_by_model = res["tables"][bset].set_index("model")["macro_auroc"]
    brow = res["boot_table"].set_index("model").loc[res["best_ensemble"]]
    summary_rows.append({
        "dataset": key, "eval_set": bset,
        "best_single": res["best_single"], "best_single_macro_auroc": macro_by_model[res["best_single"]],
        "best_ensemble": res["best_ensemble"], "best_ensemble_macro_auroc": macro_by_model[res["best_ensemble"]],
        "diff": brow["diff_vs_best_single"], "diff_ci_low": brow["diff_ci_low"], "diff_ci_high": brow["diff_ci_high"],
        "p_better": brow["p_better_than_best_single"],
    })
    method_columns[f"{key}/{bset}"] = macro_by_model

summary = pd.DataFrame(summary_rows)
by_method = pd.DataFrame(method_columns)
by_method.insert(0, "kind", ["single" if m in BACKBONES else "ensemble" for m in by_method.index])
summary.to_csv(OUTPUT_DIR / "combined_summary.csv", index=False)
by_method.to_csv(OUTPUT_DIR / "macro_auroc_by_method.csv")

print("Best single vs best ensemble (method picked on the weight sample, never on the eval set):")
display(summary.round(4))
print("\nMacro AUROC of every model / method on each dataset's main eval set:")
display(by_method.round(4))

zip_path = shutil.make_archive(str(OUTPUT_DIR.parent / "ensemble_results"), "zip",
                               root_dir=OUTPUT_DIR.parent, base_dir=OUTPUT_DIR.name)
print(f"\n{zip_path}: {Path(zip_path).stat().st_size / 1e6:.1f} MB")

Best single vs best ensemble (method picked on the weight sample, never on the eval set):


,dataset,eval_set,best_single,best_single_macro_auroc,best_ensemble,best_ensemble_macro_auroc,diff,diff_ci_low,diff_ci_high,p_better
0,chexpert,val_final,densenet121,0.7962,stack_all,0.8305,0.0341,0.0231,0.0436,1.0
1,nih,test,densenet121,0.7919,weighted_logit_per_label,0.8069,0.0150,0.0127,0.0176,1.0



Macro AUROC of every model / method on each dataset's main eval set:


,kind,chexpert/val_final,nih/test
model,,,
densenet121,single,0.7962,0.7919
resnet50,single,0.7904,0.7889
efficientnet_b0,single,0.7503,0.7922
mean_prob,ensemble,0.8081,0.8066
geo_mean_prob,ensemble,0.8123,0.8067
median_prob,ensemble,0.8064,0.8037
mean_logit,ensemble,0.8130,0.8069
rank_mean,ensemble,0.8108,0.8064
weighted_logit,ensemble,0.8142,0.8069



/kaggle/working/ensemble_results.zip: 1.5 MB
